# A11 - Ensemble Models

Zachary Fletcher

Nov 21, 2025

# Task 1

In [1]:
import pandas as pd
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import cross_validate
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.ensemble import StackingClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import VotingClassifier

import warnings
warnings.filterwarnings('ignore')

In [2]:
url = 'https://raw.githubusercontent.com/matthewpecsok/4482_fall_2022/main/data/CD_additional_modified.csv'
df = pd.read_csv(url)

In [3]:
df.head()

,age,job,marital,education,default,housing,loan,contact,month,day_of_week,...,campaign,pdays,previous,poutcome,emp_var_rate,cons_price_idx,cons_conf_idx,euribor3m,nr_employed,y
0,30,blue-collar,married,basic.9y,no,yes,no,cellular,may,fri,...,2,999,0,nonexistent,-1.8,92.893,-46.2,1.313,5099.1,no
1,39,services,single,high.school,no,no,no,telephone,may,fri,...,4,999,0,nonexistent,1.1,93.994,-36.4,4.855,5191.0,no
2,25,services,married,high.school,no,yes,no,telephone,jun,wed,...,1,999,0,nonexistent,1.4,94.465,-41.8,4.962,5228.1,no
3,38,services,married,basic.9y,no,unknown,unknown,telephone,jun,fri,...,3,999,0,nonexistent,1.4,94.465,-41.8,4.959,5228.1,no
4,47,admin.,married,university.degree,no,yes,no,cellular,nov,mon,...,1,999,0,nonexistent,-0.1,93.200,-42.0,4.191,5195.8,no


In [4]:
y_target = df.pop('y')

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4117 entries, 0 to 4116
Data columns (total 20 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   age             4117 non-null   int64  
 1   job             4117 non-null   object 
 2   marital         4117 non-null   object 
 3   education       4117 non-null   object 
 4   default         4117 non-null   object 
 5   housing         4117 non-null   object 
 6   loan            4117 non-null   object 
 7   contact         4117 non-null   object 
 8   month           4117 non-null   object 
 9   day_of_week     4117 non-null   object 
 10  duration        4117 non-null   int64  
 11  campaign        4117 non-null   int64  
 12  pdays           4117 non-null   int64  
 13  previous        4117 non-null   int64  
 14  poutcome        4117 non-null   object 
 15  emp_var_rate    4117 non-null   float64
 16  cons_price_idx  4117 non-null   float64
 17  cons_conf_idx   4117 non-null   f

In [6]:
df.describe()

,age,duration,campaign,pdays,previous,emp_var_rate,cons_price_idx,cons_conf_idx,euribor3m,nr_employed
count,4117.000000,4117.000000,4117.000000,4117.000000,4117.000000,4117.000000,4117.000000,4117.000000,4117.000000,4117.000000
mean,40.115375,256.850376,2.537042,960.403449,0.190187,0.085742,93.580131,-40.500947,3.621904,5166.496502
std,10.314847,254.749615,2.568668,191.967524,0.541765,1.562799,0.579061,4.593445,1.733448,73.670942
min,18.000000,0.000000,1.000000,0.000000,0.000000,-3.400000,92.201000,-50.800000,0.635000,4963.600000
25%,32.000000,103.000000,1.000000,999.000000,0.000000,-1.800000,93.075000,-42.700000,1.334000,5099.100000
50%,38.000000,181.000000,2.000000,999.000000,0.000000,1.100000,93.749000,-41.800000,4.857000,5191.000000
75%,47.000000,317.000000,3.000000,999.000000,0.000000,1.400000,93.994000,-36.400000,4.961000,5228.100000
max,88.000000,3643.000000,35.000000,999.000000,6.000000,1.400000,94.767000,-26.900000,5.045000,5228.100000


Some models like decision trees don't require encoding. however, given that I am going to use a combination of multiple models possibly including gradient boosting or SVMs, I will do encoding.

# Task 2

In [7]:
df_enc = pd.get_dummies(df)

In [8]:
clf = DecisionTreeClassifier().fit(df_enc, y_target)
cv_results = pd.DataFrame(cross_validate(clf, df_enc, y_target, scoring=['f1']))
display(cv_results.mean())
display(cv_results.std())

,0
fit_time,0.088360
score_time,0.002115
test_f1,NaN


,0
fit_time,0.018823
score_time,0.001343
test_f1,NaN


# Task 3

In [9]:
# from sklearn.metrics import make_scorer, f1_score

# # Define a custom F1 scorer to handle zero division issues gracefully
# # Be explicit about pos_label and average for binary classification with string labels
# f1_scorer = make_scorer(f1_score, pos_label='yes', average='binary', zero_division=1)

# # create 3 models, random forest, SVC with sigmoid kernal, and SVC with radial basis kernal.
# # Reducing parameter search space and cross-validation folds significantly to allow fitting to complete.
# rf_parameters = {'max_depth' : [5], 'n_estimators' : [1]}
# clf1 = GridSearchCV(RandomForestClassifier(random_state=42), rf_parameters, scoring=f1_scorer, refit=True, cv=2)

# svc_sig_parameters = {'C' : [1]}
# clf2 = GridSearchCV(SVC(kernel='sigmoid', random_state=42), svc_sig_parameters, scoring=f1_scorer, refit=True, cv=2)

# svc_rbf_parameters = {'C' : [1]}
# clf3 = GridSearchCV(SVC(kernel='rbf', random_state=42), svc_rbf_parameters, scoring=f1_scorer, refit=True, cv=2)

# eclf = VotingClassifier(
#     estimators=[('rf', clf1), ('svc_sig', clf2), ('svc_rbf', clf3)], voting='hard')

In [10]:
from sklearn.metrics import make_scorer, f1_score

# Define a custom F1 scorer to handle zero division issues gracefully
# Be explicit about pos_label and average for binary classification with string labels
f1_scorer = make_scorer(f1_score, pos_label='yes', average='binary', zero_division=1)

# create 3 models, random forest, SVC with sigmoid kernal, and SVC with radial basis kernal.
# Maintaining original parameter dictionaries as requested, but adding random_state and cv=2 for completion.
rf_parameters = {'max_depth' : list(range(1,5)), 'n_estimators' : [1, 10, 100]}
clf1 = GridSearchCV(RandomForestClassifier(random_state=42), rf_parameters, scoring=f1_scorer, refit=True, cv=2)

svc_sig_parameters = {'C' : list(range(1,5))}
clf2 = GridSearchCV(SVC(kernel='sigmoid', random_state=42), svc_sig_parameters, scoring=f1_scorer, refit=True, cv=2)

svc_rbf_parameters = {'C' : list(range(1,5))}
clf3 = GridSearchCV(SVC(kernel='rbf', random_state=42), svc_rbf_parameters, scoring=f1_scorer, refit=True, cv=2)

eclf = VotingClassifier(
    estimators=[('rf', clf1), ('svc_sig', clf2), ('svc_rbf', clf3)], voting='hard')

In [11]:
# # create 3 models, random forest, SVC with sigmoid kernal, and SVC with radial basis kernal.
# rf_parameters = {'max_depth' : list(range(1,5)), 'n_estimators' : [1, 10, 100]}
# clf1 = GridSearchCV(RandomForestClassifier(), rf_parameters, scoring='f1', refit=True)

# svc_sig_parameters = {'C' : list(range(1,5))}
# clf2 = GridSearchCV(SVC(kernel='sigmoid'), svc_sig_parameters, scoring='f1', refit=True)

# svc_rbf_parameters = {'C' : list(range(1,5))}
# clf3 = GridSearchCV(SVC(kernel='rbf'), svc_rbf_parameters, scoring='f1', refit=True)

# eclf = VotingClassifier(
#     estimators=[('rf', clf1), ('svc_sig', clf2), ('svc_rbf', clf3)], voting='hard')

In [12]:
# Fit the ensemble classifier. This will also fit all the GridSearchCV base estimators.
clf1.fit(df_enc, y_target)
clf2.fit(df_enc, y_target)
clf3.fit(df_enc, y_target)
eclf.fit(df_enc, y_target)

# Individual GridSearchCV estimators (clf1, clf2, clf3) are now fitted and their cv_results_ populated.

VotingClassifier(estimators=[('rf',
                              GridSearchCV(cv=2,
                                           estimator=RandomForestClassifier(random_state=42),
                                           param_grid={'max_depth': [1, 2, 3,
                                                                     4],
                                                       'n_estimators': [1, 10,
                                                                        100]},
                                           scoring=make_scorer(f1_score, response_method='predict', pos_label=yes, average=binary, zero_division=1))),
                             ('svc_sig',
                              GridSearchCV(cv=2,
                                           estimator=SVC(kernel='sigmoid',
                                                         random_state=42),
                                           param_grid={'C': [1, 2, 3, 4]},
                                           scoring=make_scorer(f1_score, response_method='predict', pos_label=yes, average=binary, zero_division=1))),
                             ('svc_rbf',
                              GridSearchCV(cv=2, estimator=SVC(random_state=42),
                                           param_grid={'C': [1, 2, 3, 4]},
                                           scoring=make_scorer(f1_score, response_method='predict', pos_label=yes, average=binary, zero_division=1)))])

In [13]:
eclf_results = pd.DataFrame(cross_validate(eclf,df_enc,y_target,scoring=f1_scorer))
display(eclf_results.head())
display(eclf_results.mean())
display(eclf_results.std())

,fit_time,score_time,test_score
0,3.498683,0.138791,0.0
1,4.710109,0.131511,0.0
2,3.485475,0.138217,0.0
3,3.447587,0.138093,0.0
4,4.089538,0.216807,0.0


,0
fit_time,3.846278
score_time,0.152684
test_score,0.000000


,0
fit_time,0.551211
score_time,0.035970
test_score,0.000000


In [14]:
clf1_results = pd.DataFrame(cross_validate(clf1,df_enc,y_target,scoring=f1_scorer))
display(clf1_results.head())
display(clf1_results.mean())
display(clf1_results.std())

,fit_time,score_time,test_score
0,2.579010,0.008662,0.485714
1,2.440578,0.008939,0.475610
2,2.299948,0.007944,0.421053
3,2.352095,0.009479,0.205607
4,3.346414,0.013801,0.460674


,0
fit_time,2.603609
score_time,0.009765
test_score,0.409732


,0
fit_time,0.428489
score_time,0.002323
test_score,0.116730


In [15]:
clf2_results = pd.DataFrame(cross_validate(clf2,df_enc,y_target,scoring=f1_scorer))
display(clf2_results.head())
display(clf2_results.mean())
display(clf2_results.std())

,fit_time,score_time,test_score
0,1.503582,0.067647,0.000000
1,1.462451,0.065788,0.241379
2,1.510195,0.065055,0.021739
3,1.458461,0.065277,0.273504
4,1.466719,0.067267,0.000000


,0
fit_time,1.480282
score_time,0.066207
test_score,0.107325


,0
fit_time,0.024575
score_time,0.001179
test_score,0.137794


In [16]:
clf3_results = pd.DataFrame(cross_validate(clf3,df_enc,y_target,scoring=f1_scorer))
display(clf3_results.head())
display(clf3_results.mean())
display(clf3_results.std())

,fit_time,score_time,test_score
0,1.231102,0.065809,0.428571
1,1.312047,0.150734,0.278689
2,1.877624,0.116278,0.360656
3,1.408216,0.073111,0.273504
4,1.208910,0.062520,0.314961


,0
fit_time,1.407580
score_time,0.093690
test_score,0.331276


,0
fit_time,0.274173
score_time,0.038525
test_score,0.064612


In [17]:
print("Best parameters found by GridSearchCV:")
print(clf1.best_params_)
print(clf2.best_params_)
print(clf3.best_params_)

Best parameters found by GridSearchCV:
{'max_depth': 3, 'n_estimators': 1}
{'C': 4}
{'C': 3}


# Task 4

In [19]:

clf1 = RandomForestClassifier(n_estimators=1, max_depth=3, random_state=1)
clf2 = SVC(kernel='sigmoid',random_state=1, C=4)
clf3 = SVC(kernel='rbf',random_state=1, C=3)

eclf = VotingClassifier(
    estimators=[('svc_sig', clf1), ('rf', clf2), ('svc_rbf', clf3)],
    voting='hard')


eclf = eclf.fit(df_enc, y_target)
pd.DataFrame(cross_validate(eclf,df_enc,y_target,scoring=f1_scorer)).agg('mean')


,0
fit_time,1.089762
score_time,0.297460
test_score,0.192532


The voting ensemble has 3 models, a random forest with one estimator and a max depth of 3, a sigmoid kernal SVC with a C of 4 and a RBF kernal SVC with a C of 3.

# Task 5

In [20]:
params = {'max_depth' : list(range(1,20,2)), 'n_estimators' :  [1, 10, 100]}

clf4 = GridSearchCV(GradientBoostingClassifier(random_state=42), params, scoring=f1_scorer, refit=True, cv=2)

In [22]:
clf4.fit(df_enc, y_target)

GridSearchCV(cv=2, estimator=GradientBoostingClassifier(random_state=42),
             param_grid={'max_depth': [1, 3, 5, 7, 9, 11, 13, 15, 17, 19],
                         'n_estimators': [1, 10, 100]},
             scoring=make_scorer(f1_score, response_method='predict', pos_label=yes, average=binary, zero_division=1))

In [24]:
results_df_clf4 = pd.DataFrame(clf4.cv_results_)
best_clf4_results = results_df_clf4[results_df_clf4['rank_test_score'] == 1]
display(best_clf4_results[['mean_test_score', 'std_test_score', 'params']].sort_values(by='mean_test_score', ascending=False))

,mean_test_score,std_test_score,params
8,0.529836,0.001265,"{'max_depth': 5, 'n_estimators': 100}"


It's similar to bagging, but all instances used for training Tr are assigned a weight. M models are then trained sequentially (in order). Each model can then be weighted. More misclassification errors suggest higher weights, and allow for the error in these particular misclassifications to be addressed for the next training.

# Task 6

In [25]:
# use same estimators in task 3 in a stack classifier

estimators = [('rf', clf1), ('svc_sig', clf2), ('svc_rbf', clf3)]

final_estimator = GradientBoostingClassifier(random_state=42)

clf5 = StackingClassifier(estimators=estimators, final_estimator=final_estimator)

In [26]:
pd.DataFrame(cross_validate(clf5,df_enc,y_target,scoring=f1_scorer)).agg('mean')

,0
fit_time,3.651128
score_time,0.236751
test_score,0.452619


basically, it combines the 3 models together, then uses the final estimator gradient boosting to try to improve the stacked model.

In [ ]:
!jupyter nbconvert --to html "file name of the notebook"